# ELEN90099: draft and final-study entry point
Authors: Yuchen Yang (1587576), Zhengnan Cao (1711701).
The recorded pilot ran on an RTX 3060 Laptop GPU. Colab runtime performance must be measured separately. Execute training cells only when starting a new campaign.


In [ ]:
from google.colab import files
import zipfile
from pathlib import Path
uploaded = files.upload()
assert "code.zip" in uploaded
with zipfile.ZipFile("code.zip") as archive:
    archive.extractall("/content/ldi_project")
%cd /content/ldi_project/src
%pip install -r requirements.txt


## Verify the bundled evidence first
This checks selected-data provenance and metadata; it does not reload omitted pilot checkpoints.


In [ ]:
!python -m unittest discover -s tests -p "test_*.py" -v
!python verify_metadata.py
import torch
print("CUDA available:", torch.cuda.is_available())


## Optional: reconstruct official data
Run only if independent reconstruction is needed. Synthetic fallback is disabled.


In [ ]:
# !python prepare_data.py


## Pilot or fuller training
Use a fresh output root for each campaign. Change epochs from 5 to 20 for the planned fuller study; doing so creates new evidence, not replacements for the submitted records.


In [ ]:
!python experiment.py --stage main --epochs 5 --seeds 42 43 44 --output-root outputs/colab_main


## Repeated sensitivities and capacity control
These are planned research runs; leave the commands commented until required.


In [ ]:
# !python experiment.py --stage window --epochs 20 --seeds 42 43 44 --output-root outputs/colab_window
# !python experiment.py --stage fraction --epochs 20 --seeds 42 43 44 --output-root outputs/colab_fraction
# !python experiment.py --stage ablation --epochs 20 --seeds 42 43 44 --output-root outputs/colab_ablation
# !python validation_search.py --dataset Electricity --model LSTM --epochs 20 --seeds 42 43 44 --output-root outputs/colab_tuning


## Summarise and independently check new runs
Saved forecasts are needed for detailed errors and checkpoint reloads.


In [ ]:
!python summarize_runs.py --root outputs/colab_main --output outputs/colab_main/summary.csv
!python verify_results.py --root outputs/colab_main --reload
!python error_analysis.py --root outputs/colab_main --output outputs/colab_main/extended
